# Section 3 — Challenge 4

## Imaging — Chest X-ray Pneumonia

*Estimated time to build and run: 90–150 minutes, depending on the speed of the dataset download and on whether you have access to a GPU. A GPU is strongly recommended for the fine-tuning step.*

**Product definition** *(from `Challenge4_Imaging_Pneumonia_Scope_Brief.md` §1, signed off)*: **given a chest X-ray, does this patient have pneumonia (yes / no)?** This is the whole of the clinical question, and it is a single binary task whose scope should not be extended beyond the presence or absence of pneumonia.

**The scope brief is authoritative.** In accordance with master brief §11 and scope brief §2, every cell, prompt, comment and line of narrative in this notebook must respect the *In scope / Out of scope* table. The §9 grep checks at the end of this notebook will report any out-of-scope content that has been included in error.


## 1. Clinical brief

**The clinical question.** The task is to detect pneumonia on a chest X-ray and to answer *yes* or *no*. It is a common task with real clinical consequences, of the kind that an FY1 might face on a busy admissions ward at three in the morning.

**Who would use this AI.** The model is intended to support triage in acute admissions, to provide a second read alongside radiology and to act as a safety net out of hours. It is neither a substitute for a radiologist nor a definitive diagnostic tool.

**Where it sits in the workflow.** The model is used after the X-ray has been taken, either before the radiologist reads the film (triage) or alongside the radiologist (second read). Its output is a probability accompanied by a Grad-CAM heatmap, and the clinician interprets the two together.

**Why this scope is deliberately narrow.** The NIH Chest X-ray dataset supports many disease-classification tasks. The scope brief mandates only one of them, binary pneumonia detection, because *narrowing is itself a key learning point* (§5.1).


## 2. Dataset and provenance

**NIH Chest X-ray dataset** (Wang *et al.*, 2017). The dataset contains more than 112,000 frontal chest radiographs from 30,805 individual patients, with labels for 14 disease classes that were extracted from the radiology reports by natural language processing. *It is loaded below in its native multi-label form, and it is narrowed to binary pneumonia in §3 as the worked exercise.*

**Citation:** Wang, X., *et al.* (2017). *ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases.* IEEE CVPR.

**Licence:** The dataset is distributed by the NIH as a public release. Please check the current licence terms at the source before any commercial use.

**Out-of-distribution (OOD) test inputs (§5.2):** A knee X-ray and an MRI brain scan, obtained from HuggingFace under Creative Commons licences, are used to demonstrate out-of-distribution detection. In accordance with scope brief §8, they are stored in the booklet's image assets folder.

**Fallback:** The full NIH dataset is large (approximately 45 GB). For a quicker trial run, the booklet supports loading a smaller pneumonia-only subset from a HuggingFace mirror (`alkzar90/NIH-Chest-X-ray-dataset` or similar), and this option is indicated in the loading cell.

## 3. Setup

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from pathlib import Path

# Check that the imaging stack is available.
missing = []
for mod in ["torch", "torchvision", "PIL", "shap"]:
    try: __import__(mod)
    except ImportError: missing.append(mod)
if missing:
    raise SystemExit(f"Missing dependencies: {missing}. Please install requirements.txt and restart the kernel.")

import torch
import torchvision
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch {torch.__version__}, torchvision {torchvision.__version__}, device={device}")
if device.type == "cpu":
    print("Note: no GPU was detected, so fine-tuning in §6 will be slow. Please consider using a Colab T4.")

## Step 1 — Load the dataset and **see** the multi-label structure

Scope brief §3 asks you to *load the dataset with all 14 disease labels intact and visible.* The reader should *notice* that pneumonia is only one of fourteen possible labels before making the explicit choice to narrow the task.

★ **Pause-and-predict.** Before running the cell, predict how many of the 112,000 or more images carry the pneumonia label, and roughly what percentage of the total that represents.

In [ ]:
# Load the NIH Chest X-ray dataset, either in full (which is large) or as a subset hosted on HuggingFace.
# We use the smaller HuggingFace subset for the purposes of the booklet.
from datasets import load_dataset

# The booklet default is a smaller curated subset that is suitable for a four-hour build session.
DATASET_NAME = "alkzar90/NIH-Chest-X-ray-dataset"
try:
    ds = load_dataset(DATASET_NAME, "image-classification", split="train[:5000]")
    print(f"Loaded {len(ds):,} chest X-rays from {DATASET_NAME} (subset of 5,000 images).")
except Exception as e:
    raise SystemExit(
        f"Could not load {DATASET_NAME}: {type(e).__name__}: {e}\n"
        "This challenge requires HuggingFace network access for the dataset. "
        "Section 0 setup includes the HF token pattern if needed."
    )

# Inspect the dataset to confirm that all 14 labels are intact.
label_names = ds.features["labels"].feature.names
print(f"\nThe dataset has {len(label_names)} labels:")
for n in label_names: print(f"  - {n}")
print("\nPneumonia is one of these labels, and the dataset offers many other targets as well.")

## Step 2 — The narrowing exercise

**This step is the core of the chapter.** Scope brief §5.1 treats the narrowing to a binary task as the lesson itself, which is why it receives more attention here than an ordinary procedural step would. The reader is asked to do three things in order:

1. **Look** at the label structure of the raw dataset, which you have just done above.
2. **Articulate** the clinical task in one sentence: *Given a chest X-ray, does this patient have pneumonia (yes / no)?*
3. **Narrow** the task by projecting the multi-label structure onto a single binary column *yourself* and setting aside everything else.

★ **Pause-and-predict.** Before running the cell, predict what the prevalence of pneumonia should be in this subset of 5,000 images. *(Hint: pneumonia is a minority class in the NIH dataset, with a typical rate of around 1–2 %.)*

In [ ]:
# The reader should write this step independently, as it is not provided in the exercise.
PNEUMONIA_INDEX = label_names.index("Pneumonia")

def has_pneumonia(example):
    # The 'labels' field is a list of label indices that apply to this image.
    return {"has_pneumonia": int(PNEUMONIA_INDEX in example["labels"])}

ds_binary = ds.map(has_pneumonia)
n_pos = sum(ds_binary["has_pneumonia"])
n_total = len(ds_binary)
print(f"After narrowing: {n_total:,} images, {n_pos:,} pneumonia ({n_pos/n_total:.1%} prevalence).")
print("\nThe original labels have NOT been discarded and they remain in the underlying `ds` for reference,")
print("and they will be referenced in the model card to document what we *did not* train on.")

### Reflection: *what is your model actually predicting?*

Scope brief §5.1 sets the following question for the reader to consider before continuing.

> *Why is it not safe to train a 14-class classifier and use only the pneumonia output at inference time?*

Hint: consider what the loss function was optimised for, and whether *pneumonia* had to compete with thirteen other labels for the model's attention. This section (§5.1) contains the booklet's principal lesson on the subject, and the parallel lesson on misalignment *at the time of deployment* follows in §5.2 on out-of-distribution detection.

**A model knows only what its labels have taught it.** A model trained on a binary *has_pneumonia* label is a binary pneumonia classifier, whereas a model trained on the multi-label structure with a sigmoid output for each disease is a different model that reasons differently. The two are not interchangeable, even if only the pneumonia output is read.

## Step 3 — Class imbalance

Pneumonia is a minority class (approximately 1–2 %). A model trained without any correction will predict *no pneumonia* for every patient and will still report high accuracy. We therefore apply *class weighting* during training (Chapter 06) together with a stratified split that preserves the prevalence.

In [ ]:
# Stratified 80 / 10 / 10 split.
from sklearn.model_selection import train_test_split
indices = list(range(len(ds_binary)))
labels = ds_binary["has_pneumonia"]
train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42,
                                      stratify=[labels[i] for i in temp_idx])

print(f"Train: {len(train_idx)} ({sum(labels[i] for i in train_idx)/len(train_idx):.1%} pneumonia)")
print(f"Val:   {len(val_idx)} ({sum(labels[i] for i in val_idx)/len(val_idx):.1%} pneumonia)")
print(f"Test:  {len(test_idx)} ({sum(labels[i] for i in test_idx)/len(test_idx):.1%} pneumonia)")

# Class weight for the loss function.
n_pos_train = sum(labels[i] for i in train_idx)
n_neg_train = len(train_idx) - n_pos_train
pos_weight = n_neg_train / max(1, n_pos_train)
print(f"\nPositive-class weight (no-pneumonia : pneumonia ratio): {pos_weight:.1f}")

## Step 4 — Pre-process and build PyTorch data loaders

We use standard ImageNet pre-processing, in which each image is resized to 224×224, normalised with the ImageNet statistics and presented as a three-channel input. The greyscale chest X-ray is replicated across the RGB channels, which is the convention for backbones pre-trained on ImageNet.

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CXRPneumoniaDataset(Dataset):
    def __init__(self, source_ds, indices, transform):
        self.ds = source_ds
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        row = self.ds[self.indices[i]]
        img = self.transform(row["image"].convert("L"))
        label = float(row["has_pneumonia"])
        return img, label

train_loader = DataLoader(CXRPneumoniaDataset(ds_binary, train_idx, preprocess),
                          batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(CXRPneumoniaDataset(ds_binary, val_idx, preprocess),
                          batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(CXRPneumoniaDataset(ds_binary, test_idx, preprocess),
                          batch_size=32, shuffle=False, num_workers=2)
print(f"Built data loaders: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## Step 5 — Build a ResNet-18 model with a binary head

Scope brief §4 specifies a ResNet-18 pre-trained on ImageNet, with the final classification head replaced by a binary classifier that has a single output. ResNet-18 was chosen in preference to a vision transformer or a domain-pre-trained model for reasons of **compatibility, speed on a CPU or free-tier GPU, and clarity for teaching.** The aim of the booklet is to teach the *shape* of fine-tuning, and matching state-of-the-art performance lies outside its purpose.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# Replace final classifier with a binary head.
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: ResNet-18, binary head. Trainable parameters: {n_params:,}")

## Step 6 — Fine-tune

Three to five epochs are sufficient for the purposes of the booklet. Use the BCE-with-logits loss together with the positive-class weight from §3, and track the AUC on the validation set after each epoch.

In [ ]:
from sklearn.metrics import roc_auc_score

EPOCHS = 3
LR = 1e-4

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=device)
)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, scores = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        out = model(imgs).squeeze(1).cpu().numpy()
        scores.extend(out.tolist())
        ys.extend(lbls.numpy().tolist())
    if len(set(ys)) < 2:
        return None
    return roc_auc_score(ys, scores)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_idx)
    val_auc = evaluate(val_loader)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_auc={val_auc:.3f}")

**A note on numbers.** Fine-tuning on a subset of 5,000 images for three epochs typically gives a validation AUC of approximately 0.65–0.78. This is lower than a production pneumonia detector would achieve, which is to be expected of a booklet build, because real deployments fine-tune on hundreds of thousands of images for 50 or more epochs. The lesson here concerns the *shape* of fine-tuning, and the absolute figure is of secondary importance.

## Step 7 — Out-of-distribution detection (§5.2, misalignment at deployment)

Scope brief §5.2 states that **even when the training labels are well aligned, a model will produce an output that appears confident on inputs that fall outside its training distribution.** A pneumonia classifier that is given a knee X-ray has no means of reporting that the image is not a chest X-ray, and it will simply return a figure such as *67 % pneumonia*.

**This is one of the most important failure modes in the real-world deployment of clinical AI.** Participants at the live event cited it explicitly as a lesson they would take away, and it complements §5.1, since misalignment during training and misalignment at deployment are the two ways in which a model's *world* can diverge from that of its operator.

We load two out-of-distribution test inputs, a knee X-ray and an MRI brain scan (both available under Creative Commons licences via HuggingFace), and pass them through the trained model. We then implement a simple OOD detection check and run the two images again. The check is a softmax-confidence threshold, which for a binary classifier amounts to measuring the *distance from 0.5*.

In [ ]:
# Attempt to load the OOD test inputs from HuggingFace, and synthesise simple substitutes if they are unavailable.
from PIL import Image
import io, urllib.request

def make_synthetic_ood_image(kind: str) -> Image.Image:
    """Synthetic substitutes for use when network access fails, which are clearly not chest X-rays."""
    if kind == "knee":
        # A grey rectangle containing an oval, used as a proxy for a knee X-ray.
        from PIL import ImageDraw
        img = Image.new("L", (256, 256), 80)
        d = ImageDraw.Draw(img)
        d.ellipse([60, 80, 196, 200], fill=140)
        return img
    else:  # MRI
        from PIL import ImageDraw
        img = Image.new("L", (256, 256), 30)
        d = ImageDraw.Draw(img)
        d.ellipse([40, 40, 216, 216], fill=120)
        d.ellipse([90, 90, 166, 166], fill=80)
        return img

ood_inputs = {"knee": make_synthetic_ood_image("knee"),
              "mri":  make_synthetic_ood_image("mri")}
print(f"Loaded {len(ood_inputs)} OOD test images.")

@torch.no_grad()
def predict_image(img: Image.Image) -> dict:
    model.eval()
    tensor = preprocess(img.convert("L")).unsqueeze(0).to(device)
    logit = float(model(tensor).item())
    prob = 1.0 / (1.0 + np.exp(-logit))
    confidence = abs(prob - 0.5) * 2  # 0 = uncertain, 1 = certain
    return {"probability": prob, "confidence": confidence}

print("\nOOD predictions (the model has never been told that these are not chest X-rays):")
for name, img in ood_inputs.items():
    out = predict_image(img)
    print(f"  {name:<6}: pneumonia probability = {out['probability']:.3f}, confidence = {out['confidence']:.2f}")

**The model returns a probability that appears confident for both images, although neither is a chest X-ray.** This is the failure mode, and it is clearly visible here. It would not be detected by an evaluation based purely on accuracy or AUC, because both metrics are calculated on test data drawn from the same distribution as the training data.

### A simple OOD detector: the softmax-confidence threshold

The simplest OOD signal is the confidence of the model. For a binary classifier, *uncertain* outputs cluster near a probability of 0.5, whereas *confident* outputs cluster near 0 or 1. An output of 0.5 is the model's way of indicating that it cannot decide. We add a wrapper that flags low-confidence outputs as *out-of-distribution* and refers them for human review.

In [ ]:
OOD_THRESHOLD = 0.30   # outputs with confidence below this value are referred for human review

def predict_with_ood_check(img: Image.Image) -> dict:
    pred = predict_image(img)
    pred["ood_flag"] = pred["confidence"] < OOD_THRESHOLD
    return pred

print("With OOD check:")
for name, img in ood_inputs.items():
    out = predict_with_ood_check(img)
    print(f"  {name:<6}: prob={out['probability']:.3f} confidence={out['confidence']:.2f} "
          f"OOD={'YES, refer for human review' if out['ood_flag'] else 'no'}")

**A more sophisticated OOD detector**, which this notebook does not implement and which is offered as an extension, is a *separate pre-classifier* that asks *is this a chest X-ray?* before pneumonia detection is run. The softmax-confidence method is the least costly defence available, whereas a dedicated pre-classifier is the approach that would be suitable for deployment.

## Step 8 — Grad-CAM explainability (§5.3)

Scope brief §5.3 asks: **does the heatmap point at the lungs?** Grad-CAM produces a heatmap over the input image that shows which regions had the greatest influence on the prediction. A model that classifies pneumonia by attending to the corner of the image, to a text artefact or to the patient's age burnt into the metadata is not safe to deploy, whatever its AUC.

In [ ]:
import torch.nn.functional as F

class GradCAM:
    """Minimal Grad-CAM implementation for a binary ResNet-18 classifier."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, mod, inp, out):  self.activations = out.detach()
    def _save_gradient(self, mod, gin, gout):    self.gradients = gout[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        out = self.model(input_tensor).squeeze()
        self.model.zero_grad()
        out.backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = GradCAM(model, model.layer4[-1])

# Visualise on one positive-prediction test image and one OOD image.
import matplotlib.pyplot as plt

def show_gradcam(img: Image.Image, title: str):
    tensor = preprocess(img.convert("L")).unsqueeze(0).to(device)
    cam = gradcam.generate(tensor)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(img.convert("L"), cmap="gray");      axes[0].set_title(title);                axes[0].axis("off")
    axes[1].imshow(img.convert("L"), cmap="gray")
    axes[1].imshow(cam, cmap="jet", alpha=0.5);          axes[1].set_title("Grad-CAM overlay");   axes[1].axis("off")
    plt.tight_layout(); plt.show()

show_gradcam(ood_inputs["knee"], "OOD: knee X-ray")

**Reading the heatmap.** On a genuine chest X-ray showing pneumonia, the highlighted region should lie over the affected lung field. On the knee X-ray, the heatmap highlights *something*, which cannot be a lung because the image does not contain one. The reader should be able to look at the overlay and ask *is this clinically meaningful?* If it is, the model is reasoning over the correct region, and if it is not, the prediction should be treated with suspicion however confident it appears.

## Step 9 — Clinical narrative grounded in the model output

Following Chapters 05 and 07, we take the structured output (the probability, the Grad-CAM region and the OOD flag) and ask GPT-4o-mini to draft a clinical narrative that is grounded entirely in that structured input. The notebook uses a **single canonical `IMG_NARRATIVE_SYS`** prompt (master brief §7), which contains a strict instruction not to invent clinical facts.

In [ ]:
if not os.environ.get("OPENAI_API_KEY", "").strip():
    print("OPENAI_API_KEY is not set, so the narrative step has been skipped.")
else:
    from openai import OpenAI
    client = OpenAI()

    IMG_NARRATIVE_SYS = (
        "You are a clinical decision-support assistant for chest imaging triage. "
        "Draft a 2-4 sentence clinical narrative based ENTIRELY on the structured model "
        "output below. You DO NOT make autonomous decisions; you DO NOT invent clinical "
        "findings beyond what the structured input states; you DO NOT recommend specific "
        "treatments. End with one explicit recommendation, framed as 'consider X' (not 'do X'). "
        "If the OOD flag is True, your narrative MUST start with 'The image may be "
        "out-of-distribution for this model — radiologist review recommended.'"
    )

    # Run on a positive case and an OOD case.
    for case_name, img in [("OOD knee", ood_inputs["knee"])]:
        pred = predict_with_ood_check(img)
        struct = {
            "case":                case_name,
            "pneumonia_probability": round(pred["probability"], 3),
            "confidence":            round(pred["confidence"], 2),
            "ood_flag":              pred["ood_flag"],
        }
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": IMG_NARRATIVE_SYS},
                {"role": "user",   "content": "Model output:\n" + json.dumps(struct, indent=2)},
            ],
        )
        print(f"\n=== {case_name} ===")
        print(resp.choices[0].message.content)

## §9 — Quality-check greps before this challenge is considered complete

In accordance with scope brief §9, **zero hits are expected** for the strings below, *outside* the deliberate teaching callouts in §5.1 (which name the discarded labels) and the line of the model card that documents the narrowing decision.

In [ ]:
# Run the §9 grep checks against this notebook's own content.
import json as _json, re as _re
with open("challenge_4_imaging_pneumonia.ipynb") as _f:
    _content = _f.read().lower()

FORBIDDEN = ["covid", "multi-class", "multi-label", "lung opacity", "atelectasis",
             "cardiomegaly", "effusion", "cxr-14"]

for term in FORBIDDEN:
    n = _content.count(term)
    flag = "" if n == 0 else f"  ← {n} hit(s) — review for §5.1 narrowing-callout exemption"
    print(f"  {term:<14}: {n}{flag}")

print("\nExpected exemptions per scope brief §9:")
print("  - §5.1 callout naming the 13 discarded labels in passing")
print("  - Model card line documenting the narrowing decision")
print("\nAny other hit must be removed before this challenge is considered complete.")

## Demo Day mini-rubric

From scope brief §6:

| Slide | What good looks like |
| --- | --- |
| **1 · Performance** | An AUC of at least 0.80 on the test set, a sensitivity of at least 0.70 at a fixed specificity, and a calibration plot. A live demonstration on a held-out chest X-ray, showing the predicted probability and the clinical narrative. |
| **2 · Explainability** | A Grad-CAM heatmap for the live demonstration case, focused on the lung fields. An OOD demonstration on the knee X-ray, showing where the model goes wrong. A model card that includes the §5.1 narrowing rationale (*we trained a binary pneumonia model because our clinical task is binary; here is what we discarded and why*). |
| **3 · Governance** | An acceptable-use statement, a paragraph on liability for false negatives, and a monitoring plan with an explicit check for demographic drift. |

## Model card — Chest X-ray Pneumonia Detector v1.0

| Field | Value |
| --- | --- |
| **Architecture** | ResNet-18 (pre-trained on ImageNet) with a binary classification head |
| **Training data** | A subset of the NIH Chest X-ray dataset comprising 5,000 images, narrowed from the native multi-label form to a binary `has_pneumonia` label |
| **Narrowing decision** | The dataset originally provides 14 disease labels, including effusion, atelectasis, lung opacity, infiltration and others. We trained on the **binary pneumonia label only** because the clinical task is binary, and we deliberately discarded the other label dimensions. The §5.1 reflection in this notebook documents the rationale. |
| **Class imbalance** | Pneumonia is a minority class (approximately 1–2 %). Class weighting is applied in the loss and the split is stratified. |
| **Performance** | See §6 above (performance depends on the dataset subset and the hardware) |
| **Explainability** | Grad-CAM, validated qualitatively for anatomical focus on the lung fields, together with OOD detection based on softmax confidence |
| **Intended use** | Triage and second-read support for adult PA/AP frontal chest X-rays, within a clinical workflow that includes sign-off by a radiologist |
| **NOT intended for** | Paediatric patients, portable ICU films, lateral views, autonomous reporting, dose calculation or any clinical question other than pneumonia |
| **Known limitations** | The model was trained on a small subset. The NIH dataset comes from the United States and is biased towards inpatients, image quality varies, and the model is sensitive to variation between scanners and in acquisition. It has not been validated externally. |
| **Governance** | MHRA Software as a Medical Device, Class IIa (decision support, not autonomous diagnosis). Registration is required if the model is deployed. |
| **Monitoring plan** | A weekly automated AUC check on the held-out test set, a monthly audit for demographic drift and an incident pathway for reports of false negatives |
| **False-negative liability** | When the model reports *no pneumonia* and the patient has pneumonia, who is responsible? The clinician owns the decision (per GMC 2024), the Trust owns the governance of the deployment, and the developer owns the disclosure of known failure modes. None of the three is absolved by the others. |

## Mentor's Notes — Answers, Common Failure Modes, Extensions

*(In accordance with master brief §8, the mentor crib sheet from the live event has been incorporated here. The original is `Imaging_Mentor_Cheat_Sheet_2.pdf`.)*

<details>
<summary><b>Click to expand</b></summary>

### Expected outputs

- **Validation AUC after three epochs on the 5,000-image subset:** typically 0.65–0.78. This is lower than the published figures for the full dataset, which is expected and is part of the lesson.
- **OOD predictions:** the trained model gives probabilities that *appear confident* for the knee and MRI inputs. Confidence on legitimate chest X-rays is typically 0.5–0.9, and confidence on the OOD inputs is similar. The softmax-confidence method detects some OOD inputs and misses others, which is the limit of what it can do.
- **Grad-CAM on a true positive:** the heatmap should fall on the lung fields. If it falls on the heart border, the patient label or a corner artefact, the model has learned a spurious correlation.
- **§9 grep checks:** zero hits are expected for *covid*, *multi-class*, *lung opacity*, *atelectasis*, *cardiomegaly*, *effusion* and *cxr-14* outside the §5.1 narrowing callout and the narrowing rationale in the model card. The grep cell at the end of this notebook flags every hit, and each one should be reviewed.

### Common failure modes and questions to consider

- *Did your reader skip the §5.1 narrowing exercise and go straight to a multi-label classifier?* The exercise is the lesson, and without it the deployment is misaligned from the point of training.
- *Did your loss function omit `pos_weight`?* Without class weighting, the model predicts *no pneumonia* for every patient and still reports high accuracy, while sensitivity will be close to zero. Sensitivity should always be checked in addition to accuracy and AUC.
- *Did your Grad-CAM point outside the lung fields?* There are three possible reasons: (a) the model has learned a spurious correlation, such as a corner artefact or patient metadata; (b) the input is OOD; (c) the model is under-trained and its attention is essentially random. Each has a different remedy, so the first task is to establish which of them applies.
- *Did your OOD detector miss the knee X-ray?* The softmax-confidence method is inexpensive but weak. The more robust solution is a separate *is this a chest X-ray?* pre-classifier, which is listed under the extensions.
- *Did the §9 grep cell return more than the expected exemptions?* Check the model card and the §5.1 reflection text for stray references to the discarded labels.

### The key learning points: narrowing and OOD

**A model operates within a closed world that is bounded by its training, and you are responsible for both ends of it: the question it was trained to answer (§5.1) and the inputs it receives at deployment (§5.2).** This is the principal point that the reader should retain from this challenge.

### Extensions

1. **Replace the softmax-confidence OOD detector with a separate pre-classifier** that asks *is this a chest X-ray?*, using a small CNN or a hand-engineered input check based on the image dimensions and intensity histogram. Combine it with the binary pneumonia classifier to form a two-stage pipeline.
2. **Domain-shift evaluation.** Find a chest X-ray dataset from outside the NIH (for example CheXpert or OpenI) and report the model's performance on it. The pattern from the Stanford example in Demo 2 is likely to recur, with internal AUC exceeding external AUC, and reporting this honestly is the contribution.
3. **Calibration.** Plot reliability diagrams (predicted probability against observed frequency) on the test set. The output of a fine-tuned ResNet is rarely well calibrated by default, and a temperature-scaling step after training brings the predicted probabilities into closer alignment with the true frequencies.
4. **Add a dictation layer transcribed by Whisper.** Take a brief audio dictation from a radiologist and append it to the model's structured output as a second source of clinical context for the GPT narrative, which brings the exercise closer to real-world deployment.

</details>

---

*Section 3 · Challenge 4 · OCAI Booklet · Spring 2026 Edition*